In [86]:
gen_report = False

In [87]:
import os
import json
import pandas as pd

root = "experiments"

records = []

for dirpath, dirnames, filenames in os.walk(root):
    if "report.json" in filenames:
        report_path = os.path.join(dirpath, "report.json")

        with open(report_path, "r") as f:
            data = json.load(f)

        record = {}

        record["VQEL"] = data.get("VQEL")

        # metrics
        for k, v in data.get("metrics", {}).items():
            record[k] = v

        # config
        for k, v in data.get("config", {}).items():
            if 'dir' in k.lower() or 'split' in k.lower():
                continue
            record[k] = v

        record["path"] = dirpath.replace('experiments/', '').replace('/results', '')
        records.append(record)

df = pd.DataFrame(records)

df["reset_unfrozen_params"] = df["reset_unfrozen_params"].map({True: "yes", False: "no"})
df["freeze_codebook"] = df["freeze_codebook"].map({True: "yes", False: "no"})
df["freeze_object_encoder"] = df["freeze_object_encoder"].map({True: "yes", False: "no"})

df.loc[df["num_iterations"] == 0, "learning_rate_tt"] = "-"
df.loc[df["agent_a_training_mode"] == 'frozen', "learning_rate_phase2_a"] = "-"
df.loc[df["best_of_n"].isna(), "best_of_n"] = 1
df.loc[df["dataset_tt"].isna(), "dataset_tt"] = "one_shape"
df.loc[df["sampling_temperature_tt"].isna(), "sampling_temperature_tt"] = df["sampling_temperature"]

df.loc[(df["num_iterations"] == 0) | (df["best_of_n"] == 1), "test_time_mode"] = "-"

df = df.where(pd.notna(df), "None")
pd.options.display.float_format = '{:.10g}'.format
df = df.map(lambda x: f"{x:.0e}" if isinstance(x, (int, float)) and x != 0 and (abs(x) < 0.01 or abs(x) == 0.1 or abs(x) == 0.01) else x)
print("Total reports loaded:", len(df))


Total reports loaded: 141


/tmp/ipykernel_953590/1318901905.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["num_iterations"] == 0, "learning_rate_tt"] = "-"
/tmp/ipykernel_953590/1318901905.py:40: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["agent_a_training_mode"] == 'frozen', "learning_rate_phase2_a"] = "-"


In [88]:
def filter_df(filters, df=df, sort_by=["seed"]):

    idx = pd.Series(True, index=df.index)

    for key, values in filters.items():
        mask = pd.Series(False, index=df.index)
        if not isinstance(values, (list, tuple, set)):
            values = [values]
        for value in values:
            if value == "!None":
                mask |= (df[key] != "None")
            else:
                mask |= (df[key] == value)
        idx &= mask 

    return df[idx].sort_values(by=sort_by)
    

In [89]:
import base64

def to_html(df):    

    # ---- highlight rule ----
    if 'mutual_play_accuracy' in df:
        max_col = 'mutual_play_accuracy'
    else:
        max_col = 'test_accuracy'
    max_val = pd.to_numeric(df[max_col]).max()

    def highlight_max_row(row):
        if  pd.to_numeric(row[max_col]) == max_val:
            return ['font-weight: bold; background-color: #ffff99'] * len(row)
        else:
            return [''] * len(row)

    # ---- style ----
    styled = (
        df.style
            .apply(highlight_max_row, axis=1)  # <<< APPLY HIGHLIGHT HERE
            .hide(axis="index")
            .format(lambda x: f"{x:.0e}" if isinstance(x, (int, float)) and x != 0 and abs(x) < 0.01 else x)
            .set_table_styles([
                {"selector": "td", "props": [("border-right", "1px solid black")]},
                {"selector": "th", "props": [
                    ("border-right", "1px solid black"),
                    ("color", "darkblue"),
                    ("font-weight", "bold"),
                    ("padding-left", "8px"),
                    ("padding-right", "8px")
                ]},
            ])
            .set_properties(**{"text-align": "center"})
    )

    html_table = styled.to_html(index=False)

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_table)

        
def add_heading(num=1, title=""):
    """Add a professional-colored heading to results.html"""
    colors = ["#0b3d91",  # dark blue
              "#800000",  # maroon
              "#205522",  # dark green
              "#4b0082",  # indigo
              "#444444"]  # dark gray
    color = colors[(num-1) % len(colors)]  # cycle through colors
    
    if gen_report:
        with open("results.html", "a") as f:
            f.write(f'<h{num} style="color:{color}; font-family:Arial, sans-serif;">{title}</h{num}>\n')
        
def clear():
    if gen_report:
        with open("results.html", "w") as f:
            f.write("")

def line():
    if gen_report:
        with open("results.html", "a") as f:
            f.write('<hr style="border:1px solid #444; margin:10px 0;">\n')
        
        
def write(text):
    html_text = text.replace("\n", "<br>\n")
    
    if gen_report:
        with open("results.html", "a") as f:
            f.write(f"{html_text}\n")


def add_plot(path="plot.png"):
    """Embed an image file directly into results.html using base64."""
    with open(path, "rb") as img:
        encoded = base64.b64encode(img.read()).decode("utf-8")

    html_img = (
        '<img src="data:image/png;base64,' +
        encoded +
        '" style="max-width:650px; height:auto;"><br>\n'
    )

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_img)
        
    if os.path.exists(path):
        os.remove(path)

In [90]:
import os
import shutil

def remove_expr(res_df):
    for path in res_df["path"]:
        full_path = os.path.join("experiments", path)

        if not os.path.isdir(full_path):
            print(f"[SKIP] Not found: {full_path}")
            continue

        answer = input(f"⚠️ Delete expr '{path}' ? (y/n): ").strip().lower()

        if answer == "y":
            shutil.rmtree(full_path)
            print(f"✅ Deleted: {path}")
        else:
            print(f"❌ Skipped: {path}")

In [91]:
from itertools import product
import pandas as pd


def extract_maxes(df, cols=["seed", "agent_a_training_mode"], max_col="test_time_training_accuracy"):
    values = []
    for col in cols:
        value = set(df[col])
        values.append(value)

    combinations = [list(x) for x in product(*values)]

    maxes = []
    for comb in combinations:
        section = filter_df({col: value for col, value in zip(cols, comb)}, df)
        if not section.empty:
            max_row = section.loc[section[max_col].idxmax()]
            maxes.append(max_row.to_frame().T)

    final_df = pd.concat(maxes, ignore_index=True)
    return final_df.sort_values(by=cols)


def mean_and_std(df, out_cols=["VQEL", "dataset", "sim", "agent_a_training_mode"]):
    rows = []

    for i in range(0, len(df), 3):
        chunk = df.iloc[i:i+3]

        mean = chunk["mutual_play_accuracy"].mean() * 100
        std = chunk["mutual_play_accuracy"].std() * 100

        rows.append({
            col: chunk[col].iloc[0] for col in out_cols} | {
            "mutual_play_accuracy": f"{mean:.1f} ± {std:.1f}"
        })

    out = pd.DataFrame(rows)
    return out.sort_values(by=out_cols)
    

In [92]:
clear()

---

In [93]:
add_heading(1, "EXP1: ")

write(
"""
"""
)

In [94]:
scaling_cols = [
    "VQEL",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "sampling_temperature_tt",
    "best_of_n",
    "test_time_training_accuracy",
    "test_time_self_play_accuracy",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
    "agent_a_training_mode", 
    "learning_rate_phase1",
    "learning_rate_phase2_b",
    "pretrained_checkpoint_a",
    "path",
]

scaling_cols_report = [
    "agent_a_training_mode",
    "learning_rate_phase1",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "self_play_accuracy_a",
    "mutual_play_accuracy",


]

adapt_cols = [
    "VQEL",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "test_time_mode",
    "num_iterations",
    "learning_rate_tt",
    "test_time_training_accuracy",
    "test_time_self_play_accuracy",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
    "agent_a_training_mode", 
    "learning_rate_phase1",
    "learning_rate_phase2_b",
    "pretrained_checkpoint_a",
    "path",
]

adapt_cols_report = [
    "agent_a_training_mode",
    "learning_rate_phase1",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
]

# Shape - Euclidean

## Backbone

In [95]:
add_heading(2, "ShapeWorld")
add_heading(3, "VQEL - Euclidean")

res = filter_df({
    "dataset": "shape",
    "sim": "euclidean",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length"])

to_html(res[scaling_cols_report])

# res[scaling_cols]

In [96]:
final = extract_maxes(res, cols=["agent_a_training_mode", "message_length"])
final[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,self_play_accuracy_a,mutual_play_accuracy,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_b,pretrained_checkpoint_a,path
0,True,shape,euclidean,"[1, 2, 3, 4]",5,one_shape,-,1e-05,1,0.765,None,0.607,0.726,frozen,1e-03,1e-03,None,20251221_0002_bs32_vocab10_repr1024_lr1_0.001_...
1,True,shape,euclidean,"[2, 3, 4]",5,one_shape,-,1e-05,1,0.746,None,0.664,0.789,frozen,1e-03,1e-03,None,20251221_0043_bs32_vocab10_repr1024_lr1_0.001_...
3,True,shape,euclidean,"[3, 4]",5,one_shape,-,1e-05,1,0.82,None,0.752,0.798,frozen,1e-03,1e-03,None,20251221_0128_bs32_vocab10_repr1024_lr1_0.001_...
2,True,shape,euclidean,[4],5,one_shape,-,1e-05,1,0.74,None,0.804,0.847,frozen,1e-03,1e-03,None,20251221_0216_bs32_vocab10_repr1024_lr1_0.001_...


## Baseline

In [97]:
add_heading(3, "")

res = filter_df({
    "dataset": "shape",
    "sim": "euclidean",
    "dataset_tt": "two_shape",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length"])

to_html(res[scaling_cols_report])

res[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,self_play_accuracy_a,mutual_play_accuracy,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_b,pretrained_checkpoint_a,path
121,True,shape,euclidean,"[1, 2, 3, 4]",5,two_shape,-,1e-01,10,0.591,0.489,0.607,0.726,frozen,1e-03,1e-04,20251221_0002_bs32_vocab10_repr1024_lr1_0.001_...,20251223_0952_bs32_vocab10_repr1024_lr1_0.001_...
129,True,shape,euclidean,"[2, 3, 4]",5,two_shape,-,1e-01,10,0.591,0.53,0.664,0.789,frozen,1e-03,1e-04,20251221_0043_bs32_vocab10_repr1024_lr1_0.001_...,20251223_0934_bs32_vocab10_repr1024_lr1_0.001_...
122,True,shape,euclidean,[4],5,two_shape,-,1e-01,10,0.612,0.539,0.804,0.847,frozen,1e-03,1e-04,20251221_0216_bs32_vocab10_repr1024_lr1_0.001_...,20251223_0935_bs32_vocab10_repr1024_lr1_0.001_...


## Scaling

In [98]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape",
    "sim": "euclidean",
    "VQEL": True,
    "test_time_mode": "scaling"
}, sort_by=["message_length", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

# res[scaling_cols]

In [99]:
final = extract_maxes(res, cols=["message_length"])
final[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,self_play_accuracy_a,mutual_play_accuracy,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_b,pretrained_checkpoint_a,path
0,True,shape,euclidean,"[1, 2, 3, 4]",5,two_shape,scaling,1e-02,30,0.592,0.495,0.607,0.726,frozen,1e-03,1e-04,20251221_0002_bs32_vocab10_repr1024_lr1_0.001_...,20251223_0218_bs32_vocab10_repr1024_lr1_0.001_...
1,True,shape,euclidean,"[2, 3, 4]",5,two_shape,scaling,1e-01,20,0.592,0.545,0.664,0.789,frozen,1e-03,1e-04,20251221_0043_bs32_vocab10_repr1024_lr1_0.001_...,20251223_0329_bs32_vocab10_repr1024_lr1_0.001_...
2,True,shape,euclidean,[4],5,two_shape,scaling,1e-02,30,0.613,0.54,0.804,0.847,frozen,1e-03,1e-04,20251221_0216_bs32_vocab10_repr1024_lr1_0.001_...,20251223_0549_bs32_vocab10_repr1024_lr1_0.001_...


## Adaptation

In [100]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape",
    "sim": "euclidean",
    "VQEL": True,
    "test_time_mode": "adaptation"
}, sort_by=["message_length", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

# res[scaling_cols]

In [101]:
final = extract_maxes(res, cols=["message_length"])
final[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,self_play_accuracy_a,mutual_play_accuracy,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_b,pretrained_checkpoint_a,path
0,True,shape,euclidean,"[1, 2, 3, 4]",5,two_shape,adaptation,1e-01,30,0.591,0.489,0.607,0.726,frozen,1e-03,1e-04,20251221_0002_bs32_vocab10_repr1024_lr1_0.001_...,20251223_0309_bs32_vocab10_repr1024_lr1_0.001_...
1,True,shape,euclidean,"[2, 3, 4]",5,two_shape,adaptation,1e-02,30,0.59,0.537,0.664,0.789,frozen,1e-03,1e-04,20251221_0043_bs32_vocab10_repr1024_lr1_0.001_...,20251223_0447_bs32_vocab10_repr1024_lr1_0.001_...
2,True,shape,euclidean,[4],5,two_shape,adaptation,1e-01,30,0.614,0.534,0.804,0.847,frozen,1e-03,1e-04,20251221_0216_bs32_vocab10_repr1024_lr1_0.001_...,20251223_0641_bs32_vocab10_repr1024_lr1_0.001_...


# Shape - Cosine

## Backbone

In [102]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length"])

to_html(res[scaling_cols_report])

# res[scaling_cols]

In [103]:
final = extract_maxes(res, cols=["message_length"])
final[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,self_play_accuracy_a,mutual_play_accuracy,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_b,pretrained_checkpoint_a,path
0,True,shape,cosine,"[1, 2, 3, 4]",5,one_shape,-,1e-05,1,0.853,None,0.684,0.838,frozen,1e-04,1e-04,None,20251220_1854_bs32_vocab10_repr1024_lr1_0.0001...
1,True,shape,cosine,"[2, 3, 4]",5,one_shape,-,1e-05,1,0.891,None,0.733,0.86,frozen,1e-04,1e-04,None,20251220_0820_bs32_vocab10_repr1024_lr1_0.0001...
3,True,shape,cosine,"[3, 4]",5,one_shape,-,1e-05,1,0.891,None,0.841,0.889,frozen,1e-03,1e-04,None,20251220_0615_bs32_vocab10_repr1024_lr1_0.001_...
2,True,shape,cosine,[4],5,one_shape,-,1e-05,1,0.862,None,0.857,0.907,frozen,1e-03,1e-04,None,20251220_1028_bs32_vocab10_repr1024_lr1_0.001_...


## Baseline

In [104]:
add_heading(3, "")

res = filter_df({
    "dataset": "shape",
    "sim": "cosine",
    "dataset_tt": "two_shape",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

res[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,self_play_accuracy_a,mutual_play_accuracy,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_b,pretrained_checkpoint_a,path
132,True,shape,cosine,"[1, 2, 3, 4]",5,two_shape,-,1e-02,30,0.514,0.511,0.684,0.838,frozen,1e-03,1e-04,20251220_1854_bs32_vocab10_repr1024_lr1_0.0001...,20251222_2114_bs32_vocab10_repr1024_lr1_0.001_...
8,True,shape,cosine,"[2, 3, 4]",5,two_shape,-,1e-01,10,0.45,0.386,0.81,0.9,frozen,1e-03,1e-04,20251220_0704_bs32_vocab10_repr1024_lr1_0.001_...,20251222_1850_bs32_vocab10_repr1024_lr1_0.001_...
140,True,shape,cosine,"[3, 4]",5,two_shape,-,1e-02,10,0.473,0.414,0.841,0.889,frozen,1e-03,1e-04,20251220_0615_bs32_vocab10_repr1024_lr1_0.001_...,20251223_0933_bs32_vocab10_repr1024_lr1_0.001_...
85,True,shape,cosine,[4],5,two_shape,-,1e-02,10,0.595,0.525,0.857,0.907,frozen,1e-03,1e-04,20251220_1028_bs32_vocab10_repr1024_lr1_0.001_...,20251223_0932_bs32_vocab10_repr1024_lr1_0.001_...


## Scaling

In [105]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "scaling"
}, sort_by=["message_length", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

# res[scaling_cols]

In [106]:
final = extract_maxes(res, cols=["message_length"])
final[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,self_play_accuracy_a,mutual_play_accuracy,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_b,pretrained_checkpoint_a,path
0,True,shape,cosine,"[1, 2, 3, 4]",5,two_shape,scaling,1e-02,10,0.527,0.52,0.684,0.838,frozen,1e-03,1e-04,20251220_1854_bs32_vocab10_repr1024_lr1_0.0001...,20251222_1916_bs32_vocab10_repr1024_lr1_0.001_...
1,True,shape,cosine,"[2, 3, 4]",5,two_shape,scaling,1e-01,20,0.463,0.421,0.81,0.9,frozen,1e-03,1e-04,20251220_0704_bs32_vocab10_repr1024_lr1_0.001_...,20251222_1121_bs32_vocab10_repr1024_lr1_0.001_...
3,True,shape,cosine,"[3, 4]",5,two_shape,scaling,1e-01,10,0.494,0.437,0.841,0.889,frozen,1e-03,1e-04,20251220_0615_bs32_vocab10_repr1024_lr1_0.001_...,20251223_0017_bs32_vocab10_repr1024_lr1_0.001_...
2,True,shape,cosine,[4],5,two_shape,scaling,1e-02,30,0.603,0.535,0.857,0.907,frozen,1e-03,1e-04,20251220_1028_bs32_vocab10_repr1024_lr1_0.001_...,20251222_2218_bs32_vocab10_repr1024_lr1_0.001_...


## Adaptation

In [107]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "adaptation"
}, sort_by=["message_length", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

# res[adapt_cols]

In [108]:
final = extract_maxes(res, cols=["message_length"])
final[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,test_time_mode,num_iterations,learning_rate_tt,test_time_training_accuracy,test_time_self_play_accuracy,self_play_accuracy_a,mutual_play_accuracy,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_b,pretrained_checkpoint_a,path
0,True,shape,cosine,"[1, 2, 3, 4]",5,adaptation,5,1e-05,0.529,0.536,0.684,0.838,frozen,1e-03,1e-04,20251220_1854_bs32_vocab10_repr1024_lr1_0.0001...,20251222_2010_bs32_vocab10_repr1024_lr1_0.001_...
1,True,shape,cosine,"[2, 3, 4]",5,adaptation,10,1e-04,0.454,0.393,0.81,0.9,frozen,1e-03,1e-04,20251220_0704_bs32_vocab10_repr1024_lr1_0.001_...,20251222_1315_bs32_vocab10_repr1024_lr1_0.001_...
3,True,shape,cosine,"[3, 4]",5,adaptation,20,1e-04,0.481,0.426,0.841,0.889,frozen,1e-03,1e-04,20251220_0615_bs32_vocab10_repr1024_lr1_0.001_...,20251223_0123_bs32_vocab10_repr1024_lr1_0.001_...
2,True,shape,cosine,[4],5,adaptation,10,1e-04,0.602,0.543,0.857,0.907,frozen,1e-03,1e-04,20251220_1028_bs32_vocab10_repr1024_lr1_0.001_...,20251222_2328_bs32_vocab10_repr1024_lr1_0.001_...


# MNIST

## Backbone

In [109]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "mnist1",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length"])

to_html(res[scaling_cols_report])

res[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,self_play_accuracy_a,mutual_play_accuracy,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_b,pretrained_checkpoint_a,path
14,True,mnist1,cosine,"[3, 4]",6,mnist2,-,1e-01,10,0.5,0.534,0.788,0.895,frozen,1e-04,1e-04,20251223_1750_bs32_vocab10_repr192_lr1_0.0001_...,20251223_1959_bs32_vocab10_repr192_lr1_0.0001_...
28,True,mnist1,cosine,"[3, 4]",2,mnist1,-,1e-01,10,0.443,0.347,0.788,0.895,frozen,1e-04,1e-04,None,20251223_1750_bs32_vocab10_repr192_lr1_0.0001_...


## Baseline

In [110]:
add_heading(3, "")

res = filter_df({
    "dataset": "mnist1",
    "sim": "cosine",
    "dataset_tt": "mnist2",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length"])

to_html(res[scaling_cols_report])

res[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,self_play_accuracy_a,mutual_play_accuracy,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_b,pretrained_checkpoint_a,path
14,True,mnist1,cosine,"[3, 4]",6,mnist2,-,1e-01,10,0.5,0.534,0.788,0.895,frozen,1e-04,1e-04,20251223_1750_bs32_vocab10_repr192_lr1_0.0001_...,20251223_1959_bs32_vocab10_repr192_lr1_0.0001_...


## Scaling

In [111]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "mnist1",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "scaling"
}, sort_by=["message_length", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

res[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,self_play_accuracy_a,mutual_play_accuracy,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_b,pretrained_checkpoint_a,path
91,True,mnist1,cosine,"[3, 4]",6,mnist2,scaling,1e-01,20,0.502,0.561,0.788,0.895,frozen,1e-04,1e-04,20251223_1750_bs32_vocab10_repr192_lr1_0.0001_...,20251223_1933_bs32_vocab10_repr192_lr1_0.0001_...
25,True,mnist1,cosine,"[3, 4]",6,mnist2,scaling,1e-01,30,0.49,0.559,0.788,0.895,frozen,1e-04,1e-04,20251223_1750_bs32_vocab10_repr192_lr1_0.0001_...,20251223_1943_bs32_vocab10_repr192_lr1_0.0001_...
137,True,mnist1,cosine,"[3, 4]",6,mnist2,scaling,1e-02,20,0.514,0.551,0.788,0.895,frozen,1e-04,1e-04,20251223_1750_bs32_vocab10_repr192_lr1_0.0001_...,20251223_1910_bs32_vocab10_repr192_lr1_0.0001_...
106,True,mnist1,cosine,"[3, 4]",6,mnist2,scaling,1e-02,30,0.518,0.551,0.788,0.895,frozen,1e-04,1e-04,20251223_1750_bs32_vocab10_repr192_lr1_0.0001_...,20251223_1919_bs32_vocab10_repr192_lr1_0.0001_...


In [112]:
final = extract_maxes(res, cols=["message_length"])
final[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,self_play_accuracy_a,mutual_play_accuracy,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_b,pretrained_checkpoint_a,path
0,True,mnist1,cosine,"[3, 4]",6,mnist2,scaling,1e-02,30,0.518,0.551,0.788,0.895,frozen,1e-04,1e-04,20251223_1750_bs32_vocab10_repr192_lr1_0.0001_...,20251223_1919_bs32_vocab10_repr192_lr1_0.0001_...


## Adaptation

In [113]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "mnist1",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "adaptation"
}, sort_by=["message_length", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

res[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,test_time_mode,num_iterations,learning_rate_tt,test_time_training_accuracy,test_time_self_play_accuracy,self_play_accuracy_a,mutual_play_accuracy,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_b,pretrained_checkpoint_a,path
9,True,mnist1,cosine,"[3, 4]",6,adaptation,5,1e-04,0.512,0.562,0.788,0.895,frozen,1e-04,1e-04,20251223_1750_bs32_vocab10_repr192_lr1_0.0001_...,20251223_1844_bs32_vocab10_repr192_lr1_0.0001_...
69,True,mnist1,cosine,"[3, 4]",6,adaptation,10,1e-04,0.511,0.551,0.788,0.895,frozen,1e-04,1e-04,20251223_1750_bs32_vocab10_repr192_lr1_0.0001_...,20251223_1902_bs32_vocab10_repr192_lr1_0.0001_...
94,True,mnist1,cosine,"[3, 4]",6,adaptation,5,1e-05,0.509,0.54,0.788,0.895,frozen,1e-04,1e-04,20251223_1750_bs32_vocab10_repr192_lr1_0.0001_...,20251223_1857_bs32_vocab10_repr192_lr1_0.0001_...
115,True,mnist1,cosine,"[3, 4]",6,adaptation,10,1e-04,0.511,0.551,0.788,0.895,frozen,1e-04,1e-04,20251223_1750_bs32_vocab10_repr192_lr1_0.0001_...,20251223_1848_bs32_vocab10_repr192_lr1_0.0001_...


In [114]:
final = extract_maxes(res, cols=["message_length"])
final[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,test_time_mode,num_iterations,learning_rate_tt,test_time_training_accuracy,test_time_self_play_accuracy,self_play_accuracy_a,mutual_play_accuracy,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_b,pretrained_checkpoint_a,path
0,True,mnist1,cosine,"[3, 4]",6,adaptation,5,1e-04,0.512,0.562,0.788,0.895,frozen,1e-04,1e-04,20251223_1750_bs32_vocab10_repr192_lr1_0.0001_...,20251223_1844_bs32_vocab10_repr192_lr1_0.0001_...
